# momentum-buffer-update — ex2: Nesterov momentum step using the buffer + (g + μ·b) form

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `momentum-buffer-update`. Running the final beacon cell reports progress against the `Optimizer: Momentum buffer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Momentum buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`momentum-buffer-update`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "momentum-buffer-update"
DD_SUBTOPIC = "Optimizer: Momentum buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Nesterov momentum — evaluate g at the lookahead point

Ex1 implemented the classical heavy-ball momentum update:
```
b ← μ·b + g            # in-place momentum buffer update
θ ← θ − lr·b           # parameter step
```

Nesterov accelerated gradient (NAG) changes ONE thing: the gradient `g` is evaluated at the LOOKAHEAD point `θ − lr·μ·b` instead of at the current `θ`. PyTorch's `SGD(nesterov=True)` collapses this into the same buffer machinery via:
```
b ← μ·b + g            # same buffer update
θ ← θ − lr·(g + μ·b)   # step uses g + μ·b instead of just b
```

**Why the `g + μ·b` form is equivalent.** Expanding the lookahead gradient via the chain rule and reorganising terms gives back exactly `g + μ·b` as the effective descent direction — without ever evaluating the gradient at a second point. Same FLOPs as plain momentum, one extra tensor add.

**Same in-place buffer contract.** The buffer list is still mutated in place (ex1's invariant). Only the formula for the per-step update direction changes.

### Exercise 2 — Nesterov momentum step using the buffer + (g + μ·b) form

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply PyTorch's Nesterov-via-buffer formulation: update each buffer in place as `b ← μ·b + g`, then compute the per-parameter descent direction as `g + μ·b` — distinct from classical momentum's plain `b`.
> Keywords: nesterov, momentum, lookahead, optimizer
> ```

**KCs targeted:** `in-place-buffer-update`, `nesterov-effective-direction-g-plus-mu-b`

Implement `ex2_nesterov_step(buffer_list, grad_list, mu)`. The Nesterov deepening of ex1's classical momentum.

Contract:

1. Mutate each `buffer_list[i]` IN PLACE: `b ← μ·b + g`. Same in-place rule as ex1 — do NOT replace the buffer object.
2. Return a NEW list of per-parameter descent directions: `d_i = g_i + μ·b_i` (after the buffer update).
3. `grad_list` is read-only — do NOT mutate the gradient tensors.
4. Lengths match: `len(buffer_list) == len(grad_list)`. (Caller guarantees this; you don't need to validate.)

Inputs:
- `buffer_list`: `list[Tensor]`, mutated in place.
- `grad_list`: `list[Tensor]`, read-only.
- `mu`: `float`, momentum coefficient.

Output: `list[Tensor]` of descent directions (same length as inputs). The CALLER applies `θ ← θ − lr·d`.

In [ ]:
def ex2_nesterov_step(buffer_list, grad_list, mu):
    out = []
    for b, g in zip(buffer_list, grad_list):
        # In-place buffer update: b ← μ·b + g
        b.mul_(mu).add_(g)
        # Nesterov descent direction: g + μ·b (post-update).
        out.append(g + mu * b)
    return out


<details><summary>Solution</summary>

```python
def ex2_nesterov_step(buffer_list, grad_list, mu):
    out = []
    for b, g in zip(buffer_list, grad_list):
        # In-place buffer update: b ← μ·b + g
        b.mul_(mu).add_(g)
        # Nesterov descent direction: g + μ·b (post-update).
        out.append(g + mu * b)
    return out
```

**`b.mul_(mu).add_(g)` is the in-place pattern.** Chains two in-place ops on the same storage. `b = mu * b + g` would allocate a new tensor and lose the in-place contract — the caller's external reference would still point at the OLD buffer.

**Why `g + μ·b` after the buffer update.** The classical descent direction is just `b` (which equals `μ·b_old + g`). The Nesterov form adds another `μ·b_post` on top — equivalent to evaluating the gradient at the lookahead point WITHOUT actually doing two forward passes.

**Grad never mutated.** The Nesterov formulation does NOT rewrite `g`. The caller can keep using the original gradient tensor for logging or gradient clipping after this call.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()